# Verificación del Entorno

**Taller ETL – Cubo SECOP Autor: Jurani Zabala Hernandez  **

**Objetivo:** Confirmar, antes de ejecutar el flujo ETL completo (`Extraccion.ipynb` -`CuboDatos.ipynb`-  `Transformacion.ipynb` - `Cargue.ipynb`), que el entorno contenerizado (JupyterLab + Spark) está correctamente disponible: que se está ejecutando dentro del contenedor Docker (no en Windows nativo), que Spark inicializa, y que se puede leer y escribir en formato Parquet.

## Verificación básica: contenedor, Spark y Parquet

In [5]:
import os
from pathlib import Path
from pyspark.sql import SparkSession

assert os.name == "posix", "Debe ejecutarse dentro del contenedor Docker."

spark = (SparkSession.builder.appName("00-Verificacion")
        .master("local[*]")
        .config("spark.sql.catalogImplementation", "in-memory")
        .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

# NOTA: se usa /tmp (filesystem nativo de Linux dentro del contenedor) en vez de
# /home/jovyan/work (carpeta montada desde Windows) para evitar el error
# "Mkdirs failed to create file" que ocurre cuando varias tareas de Spark en
# paralelo (local[*]) intentan crear carpetas temporales simultáneamente sobre
# el filesystem compartido de Docker Desktop en Windows.
ruta = Path("/tmp/prueba_parquet")
spark.createDataFrame([(1, "OK")], ["id", "estado"]).write.mode("overwrite").parquet(str(ruta))
assert spark.read.parquet(str(ruta)).first()["estado"] == "OK"
print("✓ Contenedor Linux")
print("✓ Spark:", spark.version)
print("✓ Lectura y escritura Parquet")
spark.stop()
print("✓ ENTORNO LISTO")

✓ Contenedor Linux
✓ Spark: 3.1.2
✓ Lectura y escritura Parquet
✓ ENTORNO LISTO
